In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": 1234,
    "database": "banking_project"
}

cfg = DB_CONFIG
url = f"mysql+pymysql://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['database']}"
engine = create_engine(url)

# Step 1 — Create empty fact table
create_table = """
CREATE TABLE IF NOT EXISTS fact_loan_performance (
    customer_id BIGINT,
    is_defaulted INT,
    credit_amount FLOAT,
    total_income FLOAT,
    annuity_amount FLOAT,
    credit_to_income_ratio FLOAT,
    debt_burden_ratio FLOAT,
    total_installments BIGINT,
    total_paid FLOAT,
    total_scheduled FLOAT,
    repayment_ratio FLOAT,
    late_payments_count BIGINT
)
"""

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fact_loan_performance"))
    conn.execute(text(create_table))
    conn.commit()
    print("Fact table created.")

# Step 2 — Load customers in chunks
print("Fetching customer IDs...")
customer_query = "SELECT DISTINCT sk_id_curr FROM stg_applications"
customers_df = pd.read_sql(customer_query, engine)
customer_ids = customers_df['sk_id_curr'].tolist()
print(f"Total customers: {len(customer_ids)}")

# Step 3 — Process in batches
batch_size = 5000
total_batches = len(customer_ids) // batch_size + 1
total_inserted = 0

for i in range(0, len(customer_ids), batch_size):
    batch_ids = customer_ids[i:i + batch_size]
    ids_str = ','.join(map(str, batch_ids))

    query = f"""
        SELECT
            a.sk_id_curr AS customer_id,
            a.target AS is_defaulted,
            a.amt_credit AS credit_amount,
            a.amt_income_total AS total_income,
            a.amt_annuity AS annuity_amount,
            ROUND(a.amt_credit / NULLIF(a.amt_income_total, 0), 2) AS credit_to_income_ratio,
            ROUND(a.amt_annuity / NULLIF(a.amt_income_total, 0), 2) AS debt_burden_ratio,
            COUNT(i.sk_id_prev) AS total_installments,
            SUM(i.amt_payment) AS total_paid,
            SUM(i.amt_instalment) AS total_scheduled,
            ROUND(SUM(i.amt_payment) / NULLIF(SUM(i.amt_instalment), 0), 2) AS repayment_ratio,
            SUM(CASE WHEN i.days_entry_payment > i.days_instalment THEN 1 ELSE 0 END) AS late_payments_count
        FROM stg_applications a
        LEFT JOIN stg_installments i ON a.sk_id_curr = i.sk_id_curr
        WHERE a.sk_id_curr IN ({ids_str})
        GROUP BY
            a.sk_id_curr, a.target, a.amt_credit,
            a.amt_income_total, a.amt_annuity
    """

    chunk = pd.read_sql(query, engine)
    chunk.to_sql('fact_loan_performance', engine, if_exists='append', index=False)
    total_inserted += len(chunk)

    batch_num = (i // batch_size) + 1
    print(f"  Batch {batch_num}/{total_batches} done — {total_inserted} rows inserted")

print(f"\nDone. Total rows in fact_loan_performance: {total_inserted}")

Fact table created.
Fetching customer IDs...
Total customers: 307511
  Batch 1/62 done — 5000 rows inserted
  Batch 2/62 done — 10000 rows inserted
  Batch 3/62 done — 15000 rows inserted
  Batch 4/62 done — 20000 rows inserted
  Batch 5/62 done — 25000 rows inserted
  Batch 6/62 done — 30000 rows inserted
  Batch 7/62 done — 35000 rows inserted
  Batch 8/62 done — 40000 rows inserted
  Batch 9/62 done — 45000 rows inserted
  Batch 10/62 done — 50000 rows inserted
  Batch 11/62 done — 55000 rows inserted
  Batch 12/62 done — 60000 rows inserted
  Batch 13/62 done — 65000 rows inserted
  Batch 14/62 done — 70000 rows inserted
  Batch 15/62 done — 75000 rows inserted
  Batch 16/62 done — 80000 rows inserted
  Batch 17/62 done — 85000 rows inserted
  Batch 18/62 done — 90000 rows inserted
  Batch 19/62 done — 95000 rows inserted
  Batch 20/62 done — 100000 rows inserted
  Batch 21/62 done — 105000 rows inserted
  Batch 22/62 done — 110000 rows inserted
  Batch 23/62 done — 115000 rows ins